# Final: DNS + HSMM + spillover forecasting

- the main research file: DNS factors + filtered HSMM regime features + spillover network features
- models: random walk, VARX (ridge on the 3 factors), and an XGBoost version, targets are factor levels at 1/5/20 days
- evaluation: 21 rolling windows, same convention and error format as the baseline notebooks
- run spillover.ipynb first, concepts explained in final_explained.md

In [1]:
from gaussian_hsmm import GaussianHSMM
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import xgboost as xgb
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/FRB_H15.csv").dropna()

#rename columns
df.rename(columns={"Series Description": "Date", "Market yield on U.S. Treasury securities at 1-month  constant maturity, quoted on investment basis": "0Y1M", "Market yield on U.S. Treasury securities at 3-month  constant maturity, quoted on investment basis": "0Y3M", "Market yield on U.S. Treasury securities at 6-month  constant maturity, quoted on investment basis": "0Y6M", "Market yield on U.S. Treasury securities at 1-year  constant maturity, quoted on investment basis": "1Y", "Market yield on U.S. Treasury securities at 2-year  constant maturity, quoted on investment basis": "2Y", "Market yield on U.S. Treasury securities at 3-year  constant maturity, quoted on investment basis": "3Y", "Market yield on U.S. Treasury securities at 5-year  constant maturity, quoted on investment basis": "5Y", "Market yield on U.S. Treasury securities at 7-year  constant maturity, quoted on investment basis": "7Y", "Market yield on U.S. Treasury securities at 10-year  constant maturity, quoted on investment basis": "10Y", "Market yield on U.S. Treasury securities at 20-year constant maturity, quoted on investment basis": "20Y", "Market yield on U.S. Treasury securities at 30-year  constant maturity, quoted on investment basis": "30Y"}, inplace=True)
df.rename(columns={"Market yield on U.S. Treasury securities at 20-year  constant maturity, quoted on investment basis": "20Y"}, inplace=True)

#make index to datetime for timeseries
df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace=True)

matList = ["0Y1M", "0Y3M", "0Y6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]
df = df[matList].apply(pd.to_numeric, errors="coerce").dropna()

#nelson siegel factors (built by NelsonSiegel.ipynb with maturities in years and x=0.0609)
ns = pd.read_csv("../data/ns.csv")
ns.rename(columns={"Beta 1": "level", "Beta 2": "slope", "Beta 3": "curvature"}, inplace=True)
ns["Date"] = pd.to_datetime(ns["Date"])
ns.set_index("Date", inplace=True)

#5 year breakeven inflation
BE5 = pd.read_csv("../data/T5YIE.csv").dropna()
BE5.rename(columns={"observation_date": "Date", "T5YIE": "BE5"}, inplace=True)
BE5["Date"] = pd.to_datetime(BE5["Date"])
BE5.set_index("Date", inplace=True)

#effective federal funds rate
EFFR = pd.read_csv("../data/EFFR.csv").dropna()
EFFR.rename(columns={"observation_date": "Date"}, inplace=True)
EFFR["Date"] = pd.to_datetime(EFFR["Date"])
EFFR.set_index("Date", inplace=True)

#policy spread = 2Y yield minus the overnight policy rate
rates = df[["2Y"]].merge(EFFR[["EFFR"]], left_index=True, right_index=True, how="left")
rates = rates.apply(pd.to_numeric, errors="coerce")
rates["policy_spread"] = rates["2Y"] - rates["EFFR"]

#spillover features from spillover.ipynb
spill = pd.read_csv("../data/spillover_features.csv", parse_dates=["date"])
spill.set_index("date", inplace=True)

#the same loading matrix that built ns.csv, used to turn predicted factors back into yields
maturities = [1/12, 3/12, 6/12, 1, 2, 3, 5, 7, 10, 20, 30]
x = 0.0609
A = np.array([
    [
    1,
    (1 - np.exp(-x * t)) / (x * t),
    ((1 - np.exp(-x * t)) / (x * t)) - np.exp(-x * t),
    ]
    for t in maturities
])

print("yields: ", len(df), " ns factors: ", len(ns), " spillover days: ", len(spill))

yields:  6199  ns factors:  6199  spillover days:  5947


#### HSMM regime features (filtered, no future data)

- fits the 3-state HSMM on the first 80% of the 5-feature table (same spec as hsmm.ipynb), scaler fit on train only
- causalHsmmFilter: forward filter over (regime, days remaining) pairs, day t probabilities use only data through t
- outputs per day: 3 filtered probabilities, confidence, entropy, expected days remaining

In [3]:
hsmmFeatures = ["level", "slope", "curvature", "BE5", "policy_spread"]

combined = ns.merge(BE5[["BE5"]], left_index=True, right_index=True, how="left")
combined = combined.merge(rates[["policy_spread"]], left_index=True, right_index=True, how="left")
combined = combined[hsmmFeatures].apply(pd.to_numeric, errors="coerce").dropna().sort_index()

hsmmSplit = int(0.8 * len(combined))
hsmmTrainDf = combined.iloc[:hsmmSplit]

#scaler fit only on training data
hsmmScaler = StandardScaler()
xHsmmTrain = hsmmScaler.fit_transform(hsmmTrainDf)
xHsmmAll = hsmmScaler.transform(combined)

hsmmModel = GaussianHSMM(
    n_components = 3,
    covariance_type = "diag",
    max_duration = 126,
    n_iter = 6,
    hmm_n_iter = 500,
    tol = 1e-3,
    min_covar = 1e-4,
    random_state = 42,
)
hsmmModel.fit(xHsmmTrain)
print("hsmm trained through: ", hsmmTrainDf.index[-1].date())


def gaussianLogLikelihood(observations, means, variances):
    #log probability of each observation under each regime
    logLik = np.empty((len(observations), len(means)))
    for state in range(len(means)):
        stateVar = np.maximum(variances[state], 1e-12)
        squaredDist = (observations - means[state]) ** 2 / stateVar
        logLik[:, state] = -0.5 * np.sum(np.log(2.0 * np.pi * stateVar) + squaredDist, axis=1)
    return logLik


def causalHsmmFilter(model, observations):
    #forward filter over (regime, remaining duration) pairs, day t only uses data through t
    numStates = model.n_components
    maxDuration = model.max_duration
    durations = np.arange(1, maxDuration + 1)

    logEmissions = gaussianLogLikelihood(observations, model.means_, model.covars_)

    #duration_probs_ has an unused duration zero column, slice it off
    durationProbs = model.duration_probs_[:, 1 : maxDuration + 1]

    filteredProbs = np.zeros((len(observations), numStates))
    expectedRemaining = np.zeros(len(observations))

    #before the first day: pick a start regime and a full spell length
    expanded = model.startprob_[:, None] * durationProbs

    for t in range(len(observations)):
        if t > 0:
            predicted = np.zeros((numStates, maxDuration))

            #spells with more than 1 day left lose one day
            predicted[:, :-1] = predicted[:, :-1] + expanded[:, 1:]

            #spells at 1 day left end today: move to a new regime, draw a new duration
            finishedMass = expanded[:, 0]
            enteringMass = finishedMass @ model.transmat_
            predicted = predicted + enteringMass[:, None] * durationProbs
            expanded = predicted

        #reweight by todays data fit, subtracting the max avoids number underflow
        scaledEmission = np.exp(logEmissions[t] - logEmissions[t].max())
        expanded = expanded * scaledEmission[:, None]
        total = expanded.sum()
        assert total > 0.0 and np.isfinite(total)

        expanded = expanded / total
        filteredProbs[t] = expanded.sum(axis=1)
        expectedRemaining[t] = np.sum(expanded * durations[None, :])

    return filteredProbs, expectedRemaining


filteredProbs, expectedRemaining = causalHsmmFilter(hsmmModel, xHsmmAll)
assert np.allclose(filteredProbs.sum(axis=1), 1.0)

regime = pd.DataFrame(index=combined.index)
regime["state_0_prob"] = filteredProbs[:, 0]
regime["state_1_prob"] = filteredProbs[:, 1]
regime["state_2_prob"] = filteredProbs[:, 2]
regime["state_confidence"] = filteredProbs.max(axis=1)
safeProbs = np.clip(filteredProbs, 1e-12, 1.0)
regime["state_entropy"] = -np.sum(safeProbs * np.log(safeProbs), axis=1)
regime["expected_remaining"] = expectedRemaining

regime.tail(5)

hsmm trained through:  2021-09-08


,state_0_prob,state_1_prob,state_2_prob,state_confidence,state_entropy,expected_remaining
Date,,,,,,
2026-05-08,9.972224e-25,0.999998,0.000002,0.999998,0.000026,1.000415
2026-05-11,1.633405e-20,0.985165,0.014835,0.985165,0.077192,1.619346
2026-05-12,7.939108e-19,0.617059,0.382941,0.617059,0.665486,20.873067
2026-05-13,6.711799e-24,0.999982,0.000018,0.999982,0.000215,7.375706
2026-05-14,2.531274e-23,0.999991,0.000009,0.999991,0.000116,7.354771


#### Modeling table and feature list

- joins factors, regime features, and spillover features into one aligned table, yields kept on the same dates
- defines the 35 features: 5 lags per factor, 6 regime features, 14 spillover features

In [4]:
spilloverFeatures = [
    "total_connectedness", "maximum_edge", "average_absolute_net",
    "net_0Y1M", "net_0Y3M", "net_0Y6M", "net_1Y", "net_2Y", "net_3Y",
    "net_5Y", "net_7Y", "net_10Y", "net_20Y", "net_30Y",
]
regimeFeatures = ["state_0_prob", "state_1_prob", "state_2_prob", "state_confidence", "state_entropy", "expected_remaining"]

modelDf = ns[["level", "slope", "curvature"]].merge(regime, left_index=True, right_index=True, how="inner")
modelDf = modelDf.merge(spill[spilloverFeatures], left_index=True, right_index=True, how="inner")
modelDf = modelDf.dropna().sort_index()

#yields on exactly the same dates for targets and errors
yields = df.loc[modelDf.index]

#lag features, 5 lags for each factor, then regime and spillover at the current day
features = [
    "level_1", "level_2", "level_3", "level_4", "level_5",
    "slope_1", "slope_2", "slope_3", "slope_4", "slope_5",
    "curvature_1", "curvature_2", "curvature_3", "curvature_4", "curvature_5",
]
features = features + regimeFeatures + spilloverFeatures

targets = ["target_level", "target_slope", "target_curvature"]
all_cols = features + targets

print("aligned days: ", len(modelDf), " features: ", len(features))

aligned days:  5841  features:  35


#### Supervised tables

- one table per horizon, same structure as the baseline notebooks
- row = 5 lags per factor + current-day regime and spillover features + the 3 factor levels horizon days ahead
- table row i corresponds to day i+4 in modelDf (first 4 days cannot form 5 lags)

In [5]:
horizons = [1, 5, 20]

#tables[j] is the table for horizons[j]
tables = []

for horizon in horizons:
    rows = []

    for i in range(4, len(modelDf.index) - horizon):
        lagAll = []
        #lag 1 is today, lag 2 is yesterday, matching the baseline naming
        for factor in ["level", "slope", "curvature"]:
            lag1 = modelDf[factor].iloc[i]
            lag2 = modelDf[factor].iloc[i-1]
            lag3 = modelDf[factor].iloc[i-2]
            lag4 = modelDf[factor].iloc[i-3]
            lag5 = modelDf[factor].iloc[i-4]
            lagAll.extend([lag1, lag2, lag3, lag4, lag5])

        #regime and spillover features from the current day only
        for feature in regimeFeatures:
            lagAll.append(modelDf[feature].iloc[i])
        for feature in spilloverFeatures:
            lagAll.append(modelDf[feature].iloc[i])

        #targets are the factor levels horizon days ahead
        lagAll.append(modelDf["level"].iloc[i + horizon])
        lagAll.append(modelDf["slope"].iloc[i + horizon])
        lagAll.append(modelDf["curvature"].iloc[i + horizon])

        rows.append(lagAll)

    tables.append(pd.DataFrame(rows, columns=all_cols))

print("table rows per horizon: ", len(tables[0]), len(tables[1]), len(tables[2]))

table rows per horizon:  5836 5832 5817


#### Model tuning

- fitVarxAndPredict: scaler + ridge fit on rows with known targets, predicts the 3 factors at one window
- tunes ridge alpha on 12 pseudo windows that end 20 days before the first real window (training data only)
- XGBoost settings stay fixed at the baseline values

In [6]:
def fitVarxAndPredict(table, trainEnd, originRow, alpha):
    #scaler and ridge written out step by step
    xTrain = table[features].iloc[:trainEnd]
    yTrain = table[targets].iloc[:trainEnd]
    scaler = StandardScaler()
    xTrainScaled = scaler.fit_transform(xTrain)
    model = Ridge(alpha=alpha)
    model.fit(xTrainScaled, yTrain)
    xCurrent = scaler.transform(table[features].iloc[[originRow]])
    return model.predict(xCurrent)[0]


#pseudo windows end 20 days before the first real evaluation window
firstRealWindow = len(modelDf) - 41
pseudoStarts = []
for k in range(12, 0, -1):
    pseudoStarts.append(firstRealWindow - 20 - 10 * k)

alphaGrid = [0.1, 1.0, 10.0, 100.0]
bestAlpha = []

for j in range(len(horizons)):
    horizon = horizons[j]
    table = tables[j]

    bestRmse = np.inf
    chosen = 1.0
    for alpha in alphaGrid:
        errors = []
        for start in pseudoStarts:
            trainEnd = start - horizon - 3
            predictedFactors = fitVarxAndPredict(table, trainEnd, start - 4, alpha)
            predictedYields = A @ predictedFactors
            actualYields = yields.iloc[start + horizon].to_numpy(dtype=float)
            for error in (predictedYields - actualYields):
                errors.append(error ** 2)
        rmse = np.sqrt(np.mean(errors))
        if rmse < bestRmse:
            bestRmse = rmse
            chosen = alpha
    bestAlpha.append(chosen)
    print("horizon ", horizon, " best alpha: ", chosen, " pseudo window rmse: ", round(100 * bestRmse, 2), " bp")

horizon  1  best alpha:  1.0  pseudo window rmse:  18.51  bp


horizon  5  best alpha:  1.0  pseudo window rmse:  18.52  bp


horizon  20  best alpha:  1.0  pseudo window rmse:  19.24  bp


#### Rolling window evaluation

- 21 rolling windows (baseline convention), expanding training limited to rows whose targets are known at the window start
- per window and horizon: random walk (todays yields), VARX and XGBoost factor forecasts converted to yields through the A matrix
- collects errors, predicted changes, and actual changes per model, maturity, horizon

In [7]:
#indexes for windows
windowStarts = range(len(modelDf) - 41, len(modelDf) - 20)

def rollingMetrics(errors, predictedChanges, actualChanges):
    rmse = 100 * np.sqrt(np.mean(np.array(errors)**2))
    mae = 100 * np.mean(np.abs(errors))
    directionalAccuracy = 100 * np.mean(np.sign(predictedChanges) == np.sign(actualChanges))
    return rmse, mae, directionalAccuracy


modelNames = ["Random Walk", "VARX", "XGBoost"]

#allErrors[m][i][j] holds the error list for model m, maturity i, horizon j
allErrors = []
allPredictedChanges = []
allActualChanges = []
for m in range(len(modelNames)):
    allErrors.append([[[] for _ in range(3)] for _ in range(len(matList))])
    allPredictedChanges.append([[[] for _ in range(3)] for _ in range(len(matList))])
    allActualChanges.append([[[] for _ in range(3)] for _ in range(len(matList))])

for start in windowStarts:
    for j in range(len(horizons)):
        horizon = horizons[j]
        table = tables[j]
        trainEnd = start - horizon - 3

        #VARX forecast
        predictedFactorsVarx = fitVarxAndPredict(table, trainEnd, start - 4, bestAlpha[j])
        varxYields = A @ predictedFactorsVarx

        #XGBoost forecast, one booster per factor
        predictedFactorsXgb = []
        for target in targets:
            booster = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=500, learning_rate=0.02, max_depth=3, subsample=0.8, colsample_bytree=0.8, reg_lambda=5, random_state=7)
            booster.fit(table[features].iloc[:trainEnd], table[target].iloc[:trainEnd])
            predictedFactorsXgb.append(booster.predict(table[features].iloc[[start - 4]])[0])
        xgbYields = A @ np.array(predictedFactorsXgb)

        for i in range(len(matList)):
            current = yields[matList[i]].iloc[start]
            actual = yields[matList[i]].iloc[start + horizon]

            forecasts = [current, varxYields[i], xgbYields[i]]
            for m in range(len(modelNames)):
                allErrors[m][i][j].append(forecasts[m] - actual)
                allPredictedChanges[m][i][j].append(forecasts[m] - current)
                allActualChanges[m][i][j].append(actual - current)

print("finished ", len(list(windowStarts)), " rolling windows")

finished  21  rolling windows


#### Errors

- per maturity RMSE / MAE / directional accuracy for each horizon, then the per-horizon aggregates, one block per model (baseline print format)

In [8]:
for m in range(len(modelNames)):
    print("==============================")
    print("Model: ", modelNames[m])
    print("==============================")

    #aggregate metrics to analyze error per horizon
    sum = [[0.0,0.0,0.0] for _ in range(3)]

    for i in range(len(matList)):
        print("Maturity: ", matList[i])
        for j in range(len(horizons)):
            rmse, mae, directionalAccuracy = rollingMetrics(allErrors[m][i][j], allPredictedChanges[m][i][j], allActualChanges[m][i][j])
            sum[j][0] += rmse**2
            sum[j][1] += mae
            sum[j][2] += directionalAccuracy
            print("Forecast ", horizons[j], " day, RMSE: ", rmse, " MAE: ", mae, " Directional Accuracy: ", directionalAccuracy, "%")
        print("\n")

    for j in range(len(sum)):
        for k in range(3):
            sum[j][k] = sum[j][k] / len(matList)
        sum[j][0] = np.sqrt(sum[j][0])
        print(f"Horizon: {horizons[j]} day, rmse: {sum[j][0]}, MAE: {sum[j][1]}, Directional Accuracy: {sum[j][2]} \n")

Model:  Random Walk
Maturity:  0Y1M
Forecast  1  day, RMSE:  1.4800257398019168  MAE:  1.0476190476190548  Directional Accuracy:  33.33333333333333 %
Forecast  5  day, RMSE:  3.422613871631705  MAE:  2.7619047619047707  Directional Accuracy:  14.285714285714285 %
Forecast  20  day, RMSE:  4.02373908081479  MAE:  3.4285714285714377  Directional Accuracy:  14.285714285714285 %


Maturity:  0Y3M
Forecast  1  day, RMSE:  0.9999999999999999  MAE:  0.7142857142857139  Directional Accuracy:  42.857142857142854 %
Forecast  5  day, RMSE:  1.927248223318866  MAE:  1.6190476190476248  Directional Accuracy:  14.285714285714285 %
Forecast  20  day, RMSE:  2.699206232527313  MAE:  2.3333333333333326  Directional Accuracy:  9.523809523809524 %


Maturity:  0Y6M
Forecast  1  day, RMSE:  1.4960264830861962  MAE:  1.28571428571429  Directional Accuracy:  14.285714285714285 %
Forecast  5  day, RMSE:  2.526054706642821  MAE:  1.809523809523807  Directional Accuracy:  23.809523809523807 %
Forecast  20  day